Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Modeling of Cu(111) Slab/Water Interface

Create a solid-liquid interface structure consisting of a Cu(111) slab and water molecules using ASE and pfcc_extras.

The main steps in this notebook are:
1. **Create Cu(111) slab structure:** Build a 6×8×6 slab using ASE's `fcc111` function.
2. **Create liquid structure:** Place 250 water molecules using pfcc_extras' `LiquidGenerator` and optimize the structure.
3. **Create solid-liquid interface:** Join the slab and liquid structures to generate and optimize the interface.

## Step 1-1: Import Libraries

Import the required packages.

In [ ]:
from pathlib import Path

import numpy as np
from ase.build import fcc111
from ase import Atoms
from ase import units
from ase.data import atomic_masses
from ase.io import read, write
from ase.optimize import FIRE
from tqdm.notebook import tqdm

from pfcc_extras.liquidgenerator.liquid_generator import LiquidGenerator
from pfcc_extras.structure.ase_rdkit_converter import smiles_to_atoms
from pfcc_extras.structure.surface import makesurface
from pfcc_extras.visualize import show_gui
from pfcc_extras.structure.molecule import wrap_molecule

from pfp_api_client import ASECalculator, Estimator

## Step 1-2: Create Output Directory

In [ ]:
out_dir = "output/01_modeling"

out_dir = Path(out_dir)
out_dir.mkdir(exist_ok=True, parents=True)
print(f'Output directory {out_dir} has been created.')

## Step 2: Create Cu Slab Structure

Create a Cu(111) surface slab structure using ASE's `fcc111` function.
The size is 6×8×6 (288 atoms) with a 1 Å vacuum layer on each side.

In [ ]:
# Create slab structure with (111) surface with vacuum layers of 1 Å
slab = fcc111('Cu', size=(6, 8, 6), vacuum=1, orthogonal=True)

# Get the minimum Z coordinate and shift downward by that amount
z_min = slab.positions[:, 2].min()
slab.translate([0, 0, -z_min])

# Set periodic boundary conditions
slab.pbc = (True, True, True)

# Save structure
slab_file = out_dir / 'cu_slab.cif'
slab.write(str(slab_file))
print(f'{slab_file} has been saved')

In [ ]:
# Visualize structure
show_gui(slab, representations=['ball+stick'])

## Step 3: Create Liquid Structure

Generate a liquid structure with 250 water molecules using pfcc_extras' `LiquidGenerator`.

### Step 3-1: Specify Solution Molecules, Density, and Number of Molecules

In [ ]:
# Specify solution molecules
mols = {
    'water': smiles_to_atoms('O'), # Specify water molecule using SMILES
}

# Specify density
density = 1.0

# Number of molecules in solution
numbers = {
    'water': 250
}

In [ ]:
# Count atoms in slab and solvent
num_liq = sum(len(mols[name]) * numbers[name] for name in mols)
print(f'Number of Atoms (Solid): {len(slab)}')
print(f'Number of Atoms (Liquid): {num_liq}')

### Step 3-2: Specify Liquid Structure Generation Parameters
`LiquidGenerator` is used to create the solution structure. This tool determines molecular collisions and generates structures with maximum intermolecular distances through iterative calculations.

Specify the number of iterations and the output filename below.

**note**: When roughly generating a liquid structure, it is more efficient to optimize the structure afterward rather than increasing epochs. A sufficient number of epochs is needed to prevent structural optimization from failing.

| Parameter | Description |
| :--- | :--- |
| `epochs` | Number of iterations for molecular collision detection. Recommended values vary depending on molecular complexity, typically 20-100. |
| `rough_file` | File to save the roughly generated structure |

In [ ]:
# Parameter settings
epochs = 20

# Output settings
rough_file = out_dir / 'liquid_structure.cif'

print(f'Generated structure will be saved to {rough_file} ')

### Step 3-3: Execute Structure Generation
Use `LiquidGenerator` to create the solution structure.

In [ ]:
composition = [mols[name] for name in mols for _ in range(numbers[name])]

liquid_system = Atoms()
for i in composition:
    liquid_system += i

# Determine cell shape
mass = sum(atomic_masses[i] for i in liquid_system.get_atomic_numbers())
axis_a = slab.cell[0]
axis_b = slab.cell[1]
normal_vec = np.cross(axis_a, axis_b)
area = np.linalg.norm(normal_vec)

target_vol = (mass / units.kg / density * 1e27)
unit_axis_c = slab.cell[2] / np.linalg.norm(slab.cell[2])
unit_vol = np.dot(normal_vec, unit_axis_c)
coef = target_vol / unit_vol
additional_axis_c = unit_axis_c * coef
cell = np.array([axis_a, axis_b, additional_axis_c])

cell

In [ ]:
# LiquidGenerator parameters
params = {
    "density": density,
    'cell': cell,
    'wall': True,
    "composition": composition,
    "init_structure_pos": "bottom",
    "retain_init_xy": True
}

# Run LiquidGenerator
generator = LiquidGenerator('torch', **params)
solution = generator.run(epochs=epochs)

In [ ]:
show_gui(solution, representations=['ball+stick'])

### Step 3-4: Optimize Liquid Structure

Perform structural optimization to stabilize the created liquid structure.

| Parameter | Description |
| :--- | :--- |
| `calc_mode` | Matlantis calc_mode. For solution calculations, `PBE_PLUS_D3` is often a good choice. |
| `model_version` | Matlantis model version |
| `fmax` | Convergence criterion. Calculation terminates when the maximum force on any atom falls below `fmax` [eV/Å]. |

**note**: If you plan to run simulations at around 300 K after structure creation, loosen setting of `fmax` approximately 0.2 eV/Å is still acceptable.

In [ ]:
# PFP settings
calc_mode = 'PBE_PLUS_D3'
model_version = 'v8.0.0'

# Optimization setting
fmax = 0.2

# log setting
opt_file = out_dir / 'water_opt.xyz'

In [ ]:
# Set up atoms
opt_atoms = solution.copy()
opt_atoms.calc = ASECalculator(Estimator(calc_mode=calc_mode, model_version=model_version))

# Run optimization
dyn = FIRE(opt_atoms)
dyn.run(fmax=fmax)

# Save structure
write(str(opt_file), opt_atoms)

In [ ]:
# Visualize structure
show_gui(opt_atoms, representations=['ball+stick'])

## Step 4: Create Solid-Liquid Interface
### Step 4-1: Join Solid and Liquid Structures
Create the solid-liquid interface structure by attaching the solid and liquid structures created above.

It is recommended to provide a gap (margin) between the two to prevent atoms near the boundary from being too close.
Determine the optimal margin by generating and checking the structure with the code below.

Setting the margin too wide will slightly alter the liquid density, so excessively large values should be avoided.
As a guideline, a margin that keeps the maximum force (fmax) below 10 eV/Å is recommended.

| Parameter | Description |
| :--- | :--- |
| `upper_margin` | Margin thickness between the top of the slab and the bottom of the solution [Å] |
| `lower_margin` | Margin thickness between the bottom of the slab and the top of the solution [Å] |

In [ ]:
upper_margin = 2.5
lower_margin = 2.5
interface_file = out_dir / 'cu_water_interface.cif'
interface_opt_file = out_dir / 'cu_water_interface_opt.cif'

### Step 4-2: Generate Solid-Liquid Interface Structure
Load the solid and liquid structures from files and join them together.

In [ ]:
# Get the created liquid structure
slab = read(str(slab_file))
liquid_atoms = read(str(opt_file))
wrap_molecule(liquid_atoms)

# 1. Get true slab thickness (max - min in Z direction)
slab_z_max = slab.positions[:, 2].max()
slab_z_min = slab.positions[:, 2].min()
slab_thickness = slab_z_max - slab_z_min

# 2. Get true liquid (water) thickness
liquid_z_max = liquid_atoms.positions[:, 2].max()
liquid_z_min = liquid_atoms.positions[:, 2].min()
liquid_thickness = liquid_z_max - liquid_z_min

# 3. Move liquid above slab + upper_margin
#    (Position lowest water atom at highest slab atom + margin)
shift_z = (slab_z_max + upper_margin) - liquid_z_min
liquid_atoms.positions[:, 2] += shift_z

# 4. Join structures
interface = slab + liquid_atoms

# 5. Set cell height precisely to slab + liquid thickness + margins
new_cell_z = slab_thickness + liquid_thickness + upper_margin + lower_margin
interface.cell[2, 2] = new_cell_z

# Apply periodic boundaries (wrap)
interface.wrap()

# Calculate maximum force on interface
interface.calc = ASECalculator(Estimator(calc_mode=calc_mode, model_version=model_version))
forces = interface.get_forces()
fmax = np.max(np.abs(forces))

# 構造の表示
print(f'fmax = {fmax}, A margin that keeps this value below about 10 is recommended.')
show_gui(interface)

### Step 4-3: Optimize Solid-Liquid Interface Structure

Optimize the joined interface structure to stabilize it.

In [ ]:
# Matlantis settings
calc_mode = 'PBE_PLUS_D3'
model_version = 'v8.0.0'

# Optimization setting
fmax = 0.2

opt_atoms = interface.copy()
opt_atoms.calc = ASECalculator(Estimator(calc_mode=calc_mode, model_version=model_version))

dyn = FIRE(opt_atoms)
dyn.run(fmax=fmax)

show_gui(opt_atoms, representations=['ball+stick'])

### Step 4-4: Save Solid-Liquid Interface Structure

In [ ]:
interface.write(str(interface_file))
opt_atoms.write(str(interface_opt_file))

## Next Step
The modeling of the Cu(111) slab/water interface is now complete.
In the next notebook [02_equilibrium_npzt_md_en.ipynb](./02_equilibrium_npzt_md_en.ipynb), we will equilibrate the created interface structure using MD simulation in the NPzT ensemble at 375 K.